# slices

> Atomic typed content slices — WHAT REGION of the located resource a reference consumes. The slice KIND selects the content facet on multi-facet nodes (`TimeSlice` → audio, `CharSlice` → text), which dissolves the chunk-local-vs-source-coordinate ambiguity without a frame field (pass-2 Thread 2).

In [ ]:
#| default_exp slices

## Design notes

**Architectural law (pass-2 Thread 1): slices are ATOMIC** — one contiguous span of one resource. Composition is ALWAYS edges: grouping spans (sentence/paragraph/quote, including non-contiguous and cross-graph) are higher-layer nodes with member edges, whose extent is a projection over members' atomic slices. A multi-range slice is never needed anywhere.

A typed slice is also the **"not-yet-a-node" representation** (lazy node promotion): finer granularity (words, today) lives as a slice on its parent until something references/annotates it, then it is promoted to a node. Node count stays proportional to *referenced* knowledge, not *possible* granularity.

Promoted from the zero-consumer `cjm-graph-plugin-system/utils/slices.py` sketch with renames: `TimestampSlice` → `TimeSlice`; canonical string `full_<type>` → `full:<type>`. Clean break — no legacy string parsing (pre-CR-19 graph DBs are reset, not upgraded; cross-generation read support begins only once graph content stops being cheap to reproduce from manifests + cache).

In [ ]:
#| export
import json
from dataclasses import dataclass, field
from typing import Any, ClassVar, Dict, Optional, Union

from cjm_substrate.core.errors import CapabilityInputError

In [ ]:
#| export
@dataclass(frozen=True)
class CharSlice:
    """Character-range slice into a text facet."""
    start: int  # Start character index (0-indexed, inclusive)
    end: int    # End character index (exclusive)

    KIND: ClassVar[str] = "char"  # Slice kind discriminator (selects the TEXT facet)

    def __post_init__(self):
        if self.start < 0 or self.end < self.start:
            raise CapabilityInputError(
                f"Invalid char range {self.start}-{self.end}",
                fields_invalid=["start", "end"],
            )

    def to_slice_string(self) -> str:  # e.g. "char:0-500"
        """Render the canonical slice string."""
        return f"{self.KIND}:{self.start}-{self.end}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # Wire dict with "kind" discriminator
        """Serialize to the wire dict form."""
        return {"kind": self.KIND, "start": self.start, "end": self.end}

In [ ]:
#| export
@dataclass(frozen=True)
class TimeSlice:
    """Temporal slice into an audio/video facet, in seconds.

    SINGLE-RANGE by the atomic-slice law: a fine Segment's audio ref is always
    exactly one VAD chunk; segmentation corrections move TEXT across fixed
    boundaries and never produce a multi-chunk span (where-graph-begins,
    locked 2026-06-08).
    """
    start: float  # Start time in seconds
    end: float    # End time in seconds

    KIND: ClassVar[str] = "time"  # Slice kind discriminator (selects the AUDIO facet)

    def __post_init__(self):
        if self.start < 0 or self.end < self.start:
            raise CapabilityInputError(
                f"Invalid time range {self.start}-{self.end}",
                fields_invalid=["start", "end"],
            )

    def to_slice_string(self) -> str:  # e.g. "time:6.6-9.8"
        """Render the canonical slice string."""
        return f"{self.KIND}:{self.start}-{self.end}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # Wire dict with "kind" discriminator
        """Serialize to the wire dict form."""
        return {"kind": self.KIND, "start": self.start, "end": self.end}

In [ ]:
#| export
@dataclass(frozen=True)
class FrameSlice:
    """Frame-range slice into a video facet."""
    start: int  # Start frame number (inclusive)
    end: int    # End frame number (exclusive)

    KIND: ClassVar[str] = "frame"  # Slice kind discriminator

    def __post_init__(self):
        if self.start < 0 or self.end < self.start:
            raise CapabilityInputError(
                f"Invalid frame range {self.start}-{self.end}",
                fields_invalid=["start", "end"],
            )

    def to_slice_string(self) -> str:  # e.g. "frame:0-120"
        """Render the canonical slice string."""
        return f"{self.KIND}:{self.start}-{self.end}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # Wire dict with "kind" discriminator
        """Serialize to the wire dict form."""
        return {"kind": self.KIND, "start": self.start, "end": self.end}

In [ ]:
#| export
@dataclass(frozen=True)
class LineSlice:
    """Line-range slice into code or structured text."""
    start: int  # Start line number (0-indexed, inclusive)
    end: int    # End line number (exclusive)

    KIND: ClassVar[str] = "line"  # Slice kind discriminator

    def __post_init__(self):
        if self.start < 0 or self.end < self.start:
            raise CapabilityInputError(
                f"Invalid line range {self.start}-{self.end}",
                fields_invalid=["start", "end"],
            )

    def to_slice_string(self) -> str:  # e.g. "line:10-25"
        """Render the canonical slice string."""
        return f"{self.KIND}:{self.start}-{self.end}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # Wire dict with "kind" discriminator
        """Serialize to the wire dict form."""
        return {"kind": self.KIND, "start": self.start, "end": self.end}

In [ ]:
#| export
@dataclass(frozen=True)
class PageSlice:
    """Page slice into a paginated document facet (PDF, EPUB)."""
    page: int                  # Page number (1-indexed)
    bbox: Optional[str] = None  # Optional bounding box "x1,y1,x2,y2"

    KIND: ClassVar[str] = "page"  # Slice kind discriminator

    def __post_init__(self):
        if self.page < 1:
            raise CapabilityInputError(
                f"Invalid page number {self.page}",
                fields_invalid=["page"],
            )

    def to_slice_string(self) -> str:  # e.g. "page:3" or "page:3:bbox:10,20,300,400"
        """Render the canonical slice string."""
        if self.bbox:
            return f"{self.KIND}:{self.page}:bbox:{self.bbox}"
        return f"{self.KIND}:{self.page}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # Wire dict with "kind" discriminator
        """Serialize to the wire dict form."""
        return {"kind": self.KIND, "page": self.page, "bbox": self.bbox}

In [ ]:
#| export
@dataclass(frozen=True)
class FullContent:
    """Whole-facet reference (no range) — selects a content facet without slicing.

    `SourceRef.slice=None` means "the whole resource"; `FullContent` is the
    facet-selecting variant ("the whole AUDIO of this node") for multi-facet
    targets.
    """
    content_type: str = "text"  # Facet name: "text", "audio", "image", ...

    KIND: ClassVar[str] = "full"  # Slice kind discriminator

    def to_slice_string(self) -> str:  # e.g. "full:audio"
        """Render the canonical slice string."""
        return f"{self.KIND}:{self.content_type}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # Wire dict with "kind" discriminator
        """Serialize to the wire dict form."""
        return {"kind": self.KIND, "content_type": self.content_type}

In [ ]:
#| export
@dataclass(frozen=True)
class UnknownSlice:
    """Lossless carrier for a slice kind this library version does not know.

    Same forward-compatibility law as `UnknownLocator`: round-trip, don't drop.
    """
    kind: str                                           # The unrecognized kind discriminator
    data: Dict[str, Any] = field(default_factory=dict)  # Original payload, verbatim (minus "kind")

    def to_slice_string(self) -> str:  # Best-effort canonical string: "<kind>:<canonical-json>"
        """Render a deterministic best-effort slice string."""
        canonical = json.dumps(self.data, sort_keys=True, separators=(",", ":"))
        return f"{self.kind}:{canonical}"

    def __str__(self) -> str:  # Same as `to_slice_string`
        return self.to_slice_string()

    def to_dict(self) -> Dict[str, Any]:  # The original wire dict, reconstructed verbatim
        """Serialize back to the original wire dict form."""
        return {"kind": self.kind, **self.data}

In [ ]:
#| export
TypedSlice = Union[CharSlice, TimeSlice, FrameSlice, LineSlice, PageSlice, FullContent, UnknownSlice]  # The slice sum type

SLICE_KINDS: Dict[str, type] = {  # Known-kind registry for wire-dict dispatch
    CharSlice.KIND: CharSlice,
    TimeSlice.KIND: TimeSlice,
    FrameSlice.KIND: FrameSlice,
    LineSlice.KIND: LineSlice,
    PageSlice.KIND: PageSlice,
    FullContent.KIND: FullContent,
}

In [ ]:
#| export
def slice_from_dict(
    d: Dict[str, Any]  # Wire dict with a "kind" discriminator
) -> TypedSlice:  # Typed slice; unknown kinds round-trip as UnknownSlice
    """Reconstruct a typed slice from its wire dict.

    Unknown kinds are preserved losslessly as `UnknownSlice`; known kinds are
    strict (mirror of `locator_from_dict`).
    """
    kind = d.get("kind")
    if not kind:
        raise CapabilityInputError(
            f"Slice dict missing 'kind' discriminator: {d!r}",
            fields_invalid=["kind"],
        )
    cls = SLICE_KINDS.get(kind)
    payload = {k: v for k, v in d.items() if k != "kind"}
    if cls is None:
        return UnknownSlice(kind=kind, data=payload)
    try:
        return cls(**payload)
    except TypeError as e:
        raise CapabilityInputError(
            f"Malformed '{kind}' slice payload {payload!r}: {e}",
            fields_invalid=list(payload.keys()),
        ) from e

In [ ]:
#| export
def parse_slice(
    s: str  # Canonical slice string (e.g. "char:0-500", "time:6.6-9.8", "full:audio")
) -> TypedSlice:  # Parsed typed slice
    """Parse a canonical slice string into a typed slice.

    Convenience for KNOWN kinds only (CLI args, display strings); the wire-dict
    path (`slice_from_dict`) is the forward-compatible one. Unknown kinds raise.
    """
    if s.startswith("char:"):
        a, b = s[5:].split("-", 1)
        return CharSlice(start=int(a), end=int(b))
    if s.startswith("time:"):
        a, b = s[5:].split("-", 1)
        return TimeSlice(start=float(a), end=float(b))
    if s.startswith("frame:"):
        a, b = s[6:].split("-", 1)
        return FrameSlice(start=int(a), end=int(b))
    if s.startswith("line:"):
        a, b = s[5:].split("-", 1)
        return LineSlice(start=int(a), end=int(b))
    if s.startswith("page:"):
        rest = s[5:]
        if ":bbox:" in rest:
            page_str, bbox = rest.split(":bbox:", 1)
            return PageSlice(page=int(page_str), bbox=bbox)
        return PageSlice(page=int(rest))
    if s.startswith("full:"):
        return FullContent(content_type=s[5:])
    raise CapabilityInputError(
        f"Unknown slice format: {s!r}",
        fields_invalid=["s"],
    )

In [ ]:
# canonical string + wire dict round-trips for every known kind
known = [CharSlice(0, 500), TimeSlice(6.6, 9.8), FrameSlice(0, 120),
         LineSlice(10, 25), PageSlice(3), PageSlice(3, bbox="10,20,300,400"),
         FullContent("audio")]
for sl in known:
    assert parse_slice(sl.to_slice_string()) == sl, sl
    assert slice_from_dict(sl.to_dict()) == sl, sl

assert str(TimeSlice(6.6, 9.8)) == "time:6.6-9.8"
assert str(FullContent("audio")) == "full:audio"

# unknown-kind wire round-trip
fut = {"kind": "bbox3d", "x": 1, "y": 2}
u = slice_from_dict(fut)
assert isinstance(u, UnknownSlice) and u.to_dict() == fut

# validation: inverted/negative ranges raise
for bad in (lambda: CharSlice(5, 2), lambda: TimeSlice(-1.0, 2.0), lambda: PageSlice(0)):
    try:
        bad()
        raise AssertionError("should have raised")
    except CapabilityInputError:
        pass

# legacy forms are a clean break
try:
    parse_slice("timestamp:10.5-30.0")
    raise AssertionError("should have raised")
except CapabilityInputError:
    pass